# Tarefa 4 — Exploração: Descritores Neurais com Kornia (DISK)
**Atividade 3 · TECA2 20261 · Pontos de Interesse e seus Descritores**

**Alunos:** Henryque Oliveira, Matheus Marinho e Rodrigo Oliveira

Este notebook usa a biblioteca Kornia para carregar o modelo DISK pré-treinado, extrair keypoints e descritores neurais no mesmo par de imagens da Tarefa 2 e compará-los com os métodos clássicos.

### Base teórica

- **DISK (DIscrete Keypoints)**: modelo neural que aprende detector e descritor conjuntamente por aprendizado por reforço, otimizando diretamente a qualidade do matching. Ao contrário de SIFT/ORB, que aplicam regras matemáticas fixas sobre gradientes, o DISK aprende — a partir de dados — o que constitui um bom keypoint e um bom descritor.
- **Por que neurais superam os clássicos em cenários difíceis**: mudanças drásticas de iluminação (dia/noite), diferentes estações do ano, pontos de vista muito distantes — situações em que gradientes locais mudam radicalmente, mas a semântica da cena permanece. Modelos clássicos falham porque dependem de gradientes estáveis; modelos neurais aprendem representações mais invariantes.
- **Ratio test com limiares mais altos (0.85–0.95)**: descritores neurais são mais discriminativos — há menos ambiguidade entre os dois melhores candidatos — portanto um ratio maior ainda filtra bem os falsos positivos sem descartar os verdadeiros.
- **Por que DISK e não R2D2 ou SuperPoint?** O DISK tem boa integração com Kornia e tutorial oficial disponível. R2D2 e SuperPoint resolvem o mesmo problema com filosofias diferentes; em produção a escolha depende do dataset, da plataforma e da licença.

> Referências: Torralba, Freeman, Isola (2024) Cap. 11 · Tutorial Kornia DISK · Slides TECA2 20261, slide 20.

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import kornia
import kornia.feature as KF

print('OpenCV:', cv2.__version__)
print('torch:', torch.__version__)
print('kornia:', kornia.__version__)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device)

PATH_IMG1 = 'images/ursinho1.jpeg'
PATH_IMG2 = 'images/ursinho2.jpeg'

def load(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return img, gray

def load_tensor(path, device):
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    t = torch.from_numpy(img).float() / 255.0
    return t.permute(2, 0, 1).unsqueeze(0).to(device)

OpenCV: 4.13.0
torch: 2.12.0+cpu
kornia: 0.8.3
Dispositivo: cpu
